# 01 · Read status and compare with the panel

Read supply temperature, target, loop, pump, level, units and coolant, plus raw date/time. This example calls queries only. All API temperatures are Celsius; the host UTC timestamp is separate from the potentially uncalibrated device clock.

Read [the tutorial guide](README.md) and [physical commissioning](../docs/commissioning.md) first. Use one serial owner. These notebook outputs are intentionally empty; no bench result is implied.


## Environment

From the repository root, run:

```powershell
uv sync --locked
uv run --locked --with jupyterlab --with ipykernel python -m jupyterlab
```

Choose that environment's Python kernel and run top to bottom. Restart the kernel after driver changes. The next cell opens no serial port.


In [ ]:
import sys
from pathlib import Path

# Prefer this checkout when launched from the repository root or examples/.
REPO_ROOT = Path.cwd().resolve()
for candidate in (REPO_ROOT, REPO_ROOT.parent):
    if (candidate / "src/tark_nrc/controller.py").is_file():
        REPO_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break
print(f"Python: {sys.executable}")
print(f"Repository/config base: {REPO_ROOT}")

## Imports


In [ ]:
import json
from pathlib import Path

from tark_nrc import create_chiller

## Choose connection settings

Set `CONFIG_PATH` to your verified NRC400 profile. This keeps USB adapter/unit identity bindings and verified serial framing/timeouts together. The current `[control]` section accepts only `expected_serial_number`; remove old `allow_writes`, `min_c` and `max_c` entries. The API has no write-enable switch, target range or state-based control lock. Choose targets for the actual coolant and experiment. There is no guessed COM port. `SIMULATE = True` ignores the hardware profile and opens no physical port; each run creates a fresh simulated device with no shared state. Review all operation settings before continuing.


In [ ]:
CONFIG_PATH = REPO_ROOT / "config.toml"
SIMULATE = False

## Read one complete sample and raw diagnostics, then release

Run this entire operation cell together: connection and cleanup share one context/finally block. Ctrl+C/Interrupt attempts cleanup while allowing Python to finish. Do not split the cell, kill the kernel during I/O, or rerun setup over an unfinished connection. Errors remain visible. Closing the port retains the equipment's target, loop and pump state.


In [ ]:
chiller = create_chiller(None if SIMULATE else CONFIG_PATH, simulate=SIMULATE)
with chiller:
    print(f"Backend: {chiller.transport.backend}; identity: {chiller.identity}")
    sample = chiller.read_sample()
    print(json.dumps(sample.to_dict(), indent=2))
    # Date and clock are raw diagnostics, not trusted acquisition timestamps.
    print(f"Device date (raw): {chiller.query('DAT')}")
    print(f"Device time (raw): {chiller.query('TIM')}")

print("Compare temperature, target, units, loop, pump, level and coolant with the panel.")
print("API temperatures are Celsius even when device_units is F.")
print("CTL=QUIT is retained as reported and accepted after Stop; check the panel separately.")
print("Level OK and reported pump ON do not establish physical flow or complete alarms.")
print("Connection closed; equipment state retained.")

## Check results and finish

Compare each field with the panel. CTL=QUIT is retained as a firmware status; Stop accepts S/QUIT and Run requires R. Coolant code and level do not prove actual tank contents, flow, cooling or a complete alarm state.

**Software connection closure does not stop the chiller or pump.** Follow the physical shutdown procedure separately and confirm panel state. A forced kernel termination cannot guarantee cleanup. See [the hardware test record checklist](README.md#6-record-a-hardware-test-result) for evidence to retain.
